**WARNNING:**
- Some code here are outdated, do NOT try to run the code here to reproduce the result. Instead, use ``corpusAnalysis.py``.
- But the analysis and code logistics here is correct.
- Also, this document contains some useful statistics about the dataset we conducted experiments on, check out Section 2.

# Quick Start

In [2]:
from allennlp.predictors.predictor import Predictor
import allennlp_models.coref
import spacy
spacy.load('en_core_web_sm')
nlp = spacy.load('en_core_web_sm')
predictor = Predictor.from_path("/cluster/project/infk/cotterell/ct/code/centering_exp/pretrain/coref-spanbert-large-2020.02.27.tar.gz")

In [7]:
text = "A box of magazines is under the stairs."

In [6]:
predicted = predictor.predict(document=text)
predicted

{'antecedent_indices': [[0, 1, 2, 3],
  [0, 1, 2, 3],
  [0, 1, 2, 3],
  [0, 1, 2, 3]],
 'clusters': [[[3, 3], [9, 9]]],
 'document': ['Marry',
  'believes',
  'in',
  'herself',
  'but',
  'Herny',
  'does',
  "n't",
  'believe',
  'her',
  '.'],
 'predicted_antecedents': [-1, -1, -1, 1],
 'top_spans': [[1, 1], [3, 3], [5, 5], [9, 9]]}

In [9]:
doc = nlp(text)
for token in doc:
    print(token.text, token.dep_, token.head.text, token.head.pos_,
            [child for child in token.children])
        

A det box NOUN []
box nsubj is AUX [A, of]
of prep box NOUN [magazines]
magazines pobj of ADP []
is ROOT is AUX [box, under, .]
under prep is AUX [stairs]
the det stairs NOUN []
stairs pobj under ADP [the]
. punct is AUX []


# Get Test Documents with Ontonotes Ground truth and Prediction of c2f-coref Model

## Reading ontoNotes into documents and Get Grammatical Information via Spacy
- **documents**: [\#doc, \#sent] of Sentence Instance with 11 attributes

- **clusters_info**: \#doc of defaultdict( < class 'list'\> , {42: [(16, 16), (19, 23)], 32: [(25, 27), (57, 59), (42, 44)], 116: [(82, 82), (83, 83)]})

In [ ]:
# train = pd.read_csv('X_train.csv')

In [121]:
import spacy
spacy.load('en_core_web_sm')
nlp = spacy.load('en_core_web_sm')

In [11]:
def find_span(head):
    def dfs(token):
        nonlocal start, end
        is_empty = True
        for elem in token.children:
            is_empty = False
        if is_empty:
            if token.i < start:
                start = token.i
            if token.i > end:
                end = token.i
            return
        for child in token.children:
            dfs(child)
    start, end = head.i, head.i
    dfs(head)
    return start, end

In [414]:
from allennlp_models.common.ontonotes import Ontonotes
import collections
from typing import Dict, List, Optional, Tuple, DefaultDict

file_path = "/cluster/project/infk/cotterell/ct/data/ontonotes/example.english.v4_gold_conll"
# dataset_reader = ConllCorefReader(max_span_width=64)
# instances = dataset_reader.read(file_path)
ontonotes_reader = Ontonotes()
documents = []
clusters_info = []
document_lens = []
gram_roles = []
mention_masks = []
doc_ids = []
num_utterance_with_nosrl= 0
for sentences in ontonotes_reader.dataset_document_iterator(file_path):
    clusters: DefaultDict[int, List[Tuple[int, int]]] = collections.defaultdict(list)
    gram_role_doc = []
    total_tokens = 0
#     if len(sentences[0].srl_frames) == 0:
#         #assert len(sentences[1].srl_frames) == 0
#         num_utterance_with_nosrl +=1
#         continue
    gram_role_doc = []
    mention_mask_doc = []
    doc_id_doc = []
    for sentence in sentences:
        gram_role: DefaultDict[str, List[Tuple[int, int]]] = collections.defaultdict(list)
        mention_mask = [None]*len(sentence.words)
        doc_id = [0]*len(sentence.words)
        for i, token in enumerate(sentence.words):
            doc_id[i] = total_tokens+i
        for typed_span in sentence.coref_spans: # TypedSpan = Tuple[int, Tuple[int, int]]
            # Coref annotations are on a _per sentence_
            # basis, so we need to adjust them to be relative
            # to the length of the document.
            span_id, (start, end) = typed_span
            clusters[span_id].append((start + total_tokens, end + total_tokens))
            for i, token in enumerate(sentence.words):
                if start <= i <= end:
                    mention_mask[i] = span_id
        total_tokens += len(sentence.words)
        doc = nlp(' '.join(sentence.words))
        for token in doc:
            if 'subj' in token.dep_:
                start, end = find_span(token)
                gram_role['subj'].append((start, end))
                
            if 'dobj' in token.dep_:
                start, end = find_span(token)
                gram_role['dobj'].append((start, end))
            if 'iobj' in token.dep_:
                start, end = find_span(token)
                gram_role['iobj'].append((start, end))
            if 'pobj' in token.dep_:
                start, end = find_span(token)
                gram_role['pobj'].append((start, end))             
        gram_role_doc.append(gram_role)
        mention_mask_doc.append(mention_mask)
        doc_id_doc.append(doc_id)
    gram_roles.append(gram_role_doc) # list[list[DefaultDict(str:tuple(int,int))]]
    mention_masks.append(mention_mask_doc) # list[list[list[int]]]     [[None,116,None,None,None, ...],[],[], ...]
    document_lens.append(total_tokens) # list[int] [8,12,23,24,...]
    doc_ids.append(doc_id_doc) # list[list[list[int]]]     [[0,1,2,3,...],[14,15,16,...],[], ...]
    clusters_info.append(clusters) # list[DefaultDict(int:tuple(int,int))]  [{42: [(16, 16), (19, 23)], 32: [(25, 27), (57, 59)],...},{},{}]
    documents.append(sentences) # list[list[sentence]]

The nunmber of Test_Documents:

In [20]:
import numpy as np
from allennlp_models.common.ontonotes import Ontonotes
file_path = "/cluster/work/cotterell/ct/data/ontonotes/dev.english.v4_gold_conll"
ontonotes_reader = Ontonotes()
num_tokens, num_sentences = [], []
for sentences in ontonotes_reader.dataset_document_iterator(file_path):
    num_token = sum([len(sentence.words) for sentence in sentences])
    num_sentence = len(sentences)
    num_tokens.append(num_token)
    num_sentences.append(num_sentence)

np_list= np.array(num_tokens)
print(np.mean(np_list), np.std(np_list), np.max(np_list), np.min(np_list))

np_list= np.array(num_sentences)
print(np.mean(np_list), np.std(np_list), np.max(np_list), np.min(np_list))

475.52186588921285 374.9101645008856 2314 33
27.997084548104958 26.175066423742123 127 2


In [16]:
print(gram_roles[0][1])

defaultdict(<class 'list'>, {'dobj': [(2, 3), (11, 13)], 'subj': [(5, 9), (17, 17)], 'pobj': [(15, 19)]})


## SRL Examples:

To **express** its determination , the Chinese securities regulatory department **compares** this stock reform to a die that **has** **been** **cast** .

In [63]:
sentence = documents[1][2]

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
l = []

for srl_frame in sentence.srl_frames:
    l.append([label for label in srl_frame[1]])

mention_mask = [None] * len(sentence.words)
for typed_span in sentence.coref_spans:
    span_id, (start, end) = typed_span
    for i, token in enumerate(sentence.words):
        if start <= i <= end:
            mention_mask[i] = span_id

df = pd.DataFrame(l,
                  index=[srl_frame[0] for srl_frame in sentence.srl_frames], columns=sentence.words)
df1 = pd.DataFrame([sentence.predicate_lemmas], index=['predicate_lemmas'], columns=sentence.words)
df2 = pd.DataFrame([sentence.predicate_framenet_ids], index=['framenet_ids'], columns=sentence.words)
df3 = pd.DataFrame([sentence.word_senses], index=['word_senses'], columns=sentence.words)
df4 = pd.DataFrame([sentence.pos_tags], index=['POS'], columns=sentence.words)
df5 = pd.DataFrame([sentence.named_entities], index=['named_entities'], columns=sentence.words)
df6 = pd.DataFrame([mention_mask], index=['mention_mask'], columns=sentence.words)

result = pd.concat([df, df1,df2,df3,df4,df5, df6], ignore_index=False, sort=False)
result

,Today,",",let,'s,turn,our,attention,to,a,road,cave,-,in,accident,that,happened,in,Beijing,over,the,holiday,.
let,B-ARGM-TMP,O,B-V,B-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,O
turn,O,O,O,B-ARG0,B-V,B-ARG1,I-ARG1,B-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,I-ARGM-DIR,O
happened,O,O,O,O,O,O,O,O,B-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,I-ARG1,B-R-ARG1,B-V,B-ARGM-LOC,I-ARGM-LOC,B-ARGM-TMP,I-ARGM-TMP,I-ARGM-TMP,O
predicate_lemmas,None,None,let,None,turn,None,None,None,None,None,None,None,None,None,None,happen,None,None,None,None,None,None
framenet_ids,None,None,01,None,01,None,None,None,None,None,None,None,None,None,None,01,None,None,None,None,None,None
word_senses,None,None,1,None,1,None,None,None,None,None,None,None,None,None,None,1,None,None,None,None,None,None
POS,NN,",",VB,PRP,VB,PRP$,NN,IN,DT,NN,NN,HYPH,NN,NN,WDT,VBD,IN,NNP,IN,DT,NN,.
named_entities,B-DATE,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,B-GPE,O,O,O,O
mention_mask,40,None,None,None,None,None,None,None,67,67,67,67,67,67,67,67,67,110,67,75,75,None


In [131]:
doc = nlp(' '.join(documents[0][1].words))
for chunk in doc.noun_chunks:
    print(chunk.text, chunk.root.text, chunk.root.dep_,
            chunk.root.head.text)
gram_role_mask = [None]*len(documents[0][1].words)
for i, word in enumerate(documents[0][1].words):
    for chunk in doc.noun_chunks:
        if word in chunk.text:
            gram_role_mask[i] = chunk.root.dep_
            continue
df7 = pd.DataFrame([gram_role_mask], index=['gram_role'], columns=documents[0][1].words)

its determination determination dobj express
the Chinese securities regulatory department department nsubj compares
this stock reform reform dobj compares
a die die pobj to


In [60]:
for i, document in enumerate(documents):
    print("==========START================")
    text = ""
    for j, sentence in enumerate(document):
        print(j,end=' ')
        for k, word in enumerate(sentence.words):
            if mention_masks[i][j][k] is not None:
                print(word.upper(),end=' ')
            else:
                print(word ,end=' ')
        print('\n',end='')
    print("===========END===============")

==========START================
0 -- basically , it was unanimously agreed upon by the various relevant parties . 
1 To express ITS determination , THE CHINESE SECURITIES REGULATORY DEPARTMENT compares THIS STOCK REFORM to a die that has been cast . 
2 It takes time to prove whether THE STOCK REFORM can really meet expectations , and whether any deviations that arise during THE STOCK REFORM can be promptly corrected . 
3 Dear viewers , the China News program will end here . 
4 This is Xu Li . 
5 Thank YOU EVERYONE for watching . 
6 Coming up is the Focus Today program hosted by Wang Shilin . 
7 Good-bye , dear viewers . 
===========END===============
==========START================
0 Hello , dear viewers . 
1 Welcome to FOCUS TODAY . 
2 TODAY , let 's turn our attention to A ROAD CAVE - IN ACCIDENT THAT HAPPENED IN BEIJING OVER THE HOLIDAY . 
3 Before dawn on JANUARY 3 , a sewage pipe leakage accident occurred at the main and side roads of JINGGUANG BRIDGE , EAST THIRD RING ROAD , BEIJ

## Get Prediction of c2f-coref Model
- **predicted_dicts**: [#Doc, #Sent] of dict('top_spans', 'antecedent_indices', 'predicted_antecedents', 'document', 'clusters')
- 'top_spans': list of pairs (start,end)
- 'antecedent_indices'
- 'predicted_antecedents'
- 'document'
- 'clusters

In [359]:
predicted_dicts = []
for document in documents:
    text = ""
    for sentence in document:
        text += ' '+' '.join(map(str, sentence.words))
    predicted_dict = predictor.predict(document=text)
    predicted_dicts.append(predicted_dict)

Now we have **documents**, **clusters_info** (ground true), **predicted_dicts** (contains predicted cluster)

In [360]:
len(predicted_dicts)

2

# Ground Truth Analysis

## Compare naive metrics of coherence based on Centering Theory

### GOAL
- Identify which of the Centering-based metrics (of coherence) represent the most promising candidates for text structuring

- To analysis how successful Centering’s constraints are *on their own* in generating a felicitous text structure.


### Methodology
In this work, we explored this question by developing a search-based approach (ranking all candidate orderings of clauses by a given metric to get the best-scoring ordering) to text structuring purely based on Centering, in which the role of other factors is deliberately ignored. 

The constraints of Centering act as ranking factors.

We are using corpus-based methods instead of generally more expensive psycholinguistic techniques.


### Input
we assume that the input to text structuring is **a set of clauses**.

### Output
The output of text structuring is merely **an ordering of these clauses**, rather than the tree-like structure of database facts often used in traditional deep generation.


### The Parameters (the definitions of the central notions) of Centering:
- ''utterances``: sentences in OntoNotes(instead of finite unit in the gnome corpus)
- ''CF-list elements`` (candidate entities): clusters_ids / clusters_ids + sub + obj + named entities + ARG0&ARG1 / clusters_ids +  predicted_ccluster_ids
- ''CF-list Ranking``: 
    - Grammatical Role: Pronoun (POS) > Subject > Object (Spacy Dependency)
    - Semanntic Role: Pronoun (POS) > Subject > Object (Ground Truth) ?


## Code

### The definition of **CenteringUtterance**:

In [145]:
from enum import Enum
import collections
from allennlp_models.common.ontonotes import OntonotesSentence
from allennlp.data.dataset_readers.dataset_utils.span_utils import TypedSpan
from typing import Dict, List, Optional, Tuple, DefaultDict


class Transition(Enum):
    NA = 0
    RETAIN = 1
    CONTIUNE = 2
    S_SHIFT = 3
    R_SHIFT = 4
    NOCB = 5


class CenteringUtterance:
    def __init__(self, sentence: OntonotesSentence, sentence_id, gram_role=None,
                 CB=None, transition=Transition.NA, cheapness=None, coherence=None, salience=None) -> None:
        self.document_id = sentence.document_id
        self.sentence_id = sentence_id
        self.words = sentence.words
        self.candidate_entities = self._set_candidate_entities(sentence.coref_spans)
        self.CF_list = self._set_CF_list(sentence.pos_tags, gram_role=gram_role)
        self.CB = CB
        self.transition = transition
        self.cheapness = cheapness
        self.coherence = coherence
        self.salience = salience

    def _set_candidate_entities(self, coref_spans, gram_role_mask=None, mention_mask=None, doc_id=None):
        return coref_spans

    def _set_CF_list(self, postag, gram_role=None, mention_mask=None, doc_id=None):
        entities: DefaultDict[str, List[TypedSpan]] = collections.defaultdict(list)
        for typed_span in self.candidate_entities:
            span_id, span = typed_span
            if len(span) == 1 and ("PRP" in postag[span[0]]):
                entities["pronuons"].append(typed_span)
                continue
            if span in gram_role["subj"]:
                entities["subj"].append(typed_span)
                continue
            if span in gram_role["dobj"]:
                entities["dobj"].append(typed_span)
                continue
            if span in gram_role["iobj"]:
                entities["iobj"].append(typed_span)
                continue
            if span in gram_role["pobj"]:
                entities["pobj"].append(typed_span)
                continue
            else:
                entities["others"].append(typed_span)
        return entities["pronuons"] + entities["subj"] + entities["dobj"] + entities["iobj"] + entities["pobj"] + entities["others"]

    def set_CB_and_cheapness_coherence(self, centering_document):
        '''
        Unlike the original version of CT,
        we decide to compare Un to the nearest previous Utterance with a non-empty Cf-list
        :param centering_document:
        :return:
        '''
        # Seeking the nearest valid sentence (we are skipping utterances with empty candidate_entities, e.g. "Uh-huh.")
        for i in range(self.sentence_id-1, -1,-1):
            previousU = centering_document[i]
            if len(previousU.CF_list):
                break
        # if no previous sentences are valid (candidate_entities/CF_list is not empty),
        # we return with CB and cheapness&coherence being None.
        if len(previousU.CF_list) == 0 or len(self.candidate_entities) == 0:
            return
        # set CB as the top ranking entity in previousU.CF_list which exists in current candidate_entities
        tmp_list = [entity[0] for entity in self.candidate_entities]
        if  previousU.CF_list[0][0] in tmp_list:
            self.CB = previousU.CF_list[0]
            self.cheapness = True
            if previousU.CB and self.CB[0]!=previousU.CB[0]:
                self.coherence = False
            else:
                self.coherence = True
            return
        # if CB is not equal to the top element in previousU.CF_list (CP_{n-1})
        self.cheapness = False
        # continue searching other elements in previousU.CF_list
        for typed_span in previousU.CF_list:
            if typed_span[0] in tmp_list :
                self.CB = typed_span
                if previousU.CB and self.CB[0] != previousU.CB[0]:
                    self.coherence = False
                else:
                    self.coherence = True
                return
        # if candidate_entities is not empty and previousU.CF_list is not empty but they are disjoint, then set CB to NOCB
        self.coherence = False

    def set_salience(self):
        if self.CF_list != []:
            if not self.CB:
                self.salience = False
            else:
                self.salience = (self.CB[0]==self.CF_list[0][0])

    def set_transition(self):
        if self.coherence == True and self.salience==True:
            self.transition = Transition.CONTIUNE
        elif self.coherence == True and self.salience==False:
            self.transition = Transition.RETAIN
        elif self.coherence == False and self.salience==True:
            self.transition = Transition.S_SHIFT
        elif self.coherence == False and self.salience==False:
            self.transition = Transition.R_SHIFT
            

In [30]:
print(gram_roles[0][1])
print(documents[0][0].words)
centeringUtterance = CenteringUtterance(documents[0][1], gram_role=gram_roles[0][1])
centeringUtterance.candidate_entities


defaultdict(<class 'list'>, {'dobj': [(2, 3), (11, 13)], 'subj': [(5, 9), (17, 17)], 'pobj': [(15, 19)], 'iobj': []})
['--', 'basically', ',', 'it', 'was', 'unanimously', 'agreed', 'upon', 'by', 'the', 'various', 'relevant', 'parties', '.']


{(32, (11, 13)), (42, (2, 2)), (42, (5, 9))}

### Get Table 1 and Table 2 (Example):

In [299]:
centering_documents = []
for i, document in enumerate(documents):
    centeringUtterance = CenteringUtterance(document[0], 0, gram_role=gram_roles[i][0])
    centering_document = [centeringUtterance]
    for j in range(1,len(document)):
        prviousUtterance = centeringUtterance
        centeringUtterance = CenteringUtterance(document[j], j, gram_role=gram_roles[i][j])
        centeringUtterance.set_CB_and_cheapness_coherence(centering_document)
        centeringUtterance.set_salience()
        centeringUtterance.set_transition()
        centering_document.append(centeringUtterance)
        if len(centeringUtterance.candidate_entities) != 0:
            print(centeringUtterance.CB, centeringUtterance.transition, centeringUtterance.cheapness, centeringUtterance.coherence, centeringUtterance.salience,centeringUtterance.CF_list)
    print("=========================")
    centering_documents.append(centering_document)

None Transition.NA None None False [(42, (5, 9)), (32, (11, 13)), (42, (2, 2))]
(32, (11, 13)) Transition.CONTIUNE False True True [(32, (6, 8)), (32, (21, 23))]
None Transition.R_SHIFT False False False [(116, (1, 1)), (116, (2, 2))]
None Transition.NA None None False [(64, (2, 3))]
None Transition.R_SHIFT False False False [(75, (19, 20)), (110, (17, 17)), (67, (8, 20)), (40, (0, 0))]
(110, (17, 17)) Transition.RETAIN False True False [(67, (32, 35)), (28, (3, 4)), (110, (27, 28)), (120, (19, 28))]
(110, (27, 28)) Transition.CONTIUNE False True True [(110, (3, 4))]
None Transition.R_SHIFT False False False [(75, (15, 16))]
(75, (15, 16)) Transition.CONTIUNE True True True [(75, (1, 2))]
None Transition.R_SHIFT False False False [(53, (5, 7)), (117, (9, 10)), (40, (11, 11))]
(53, (5, 7)) Transition.RETAIN True True False [(117, (7, 8)), (5, (10, 24)), (53, (2, 5))]
None Transition.R_SHIFT False False False [(106, (8, 11)), (84, (2, 11))]
None Transition.R_SHIFT False False False [(53,

### Some issue:
1. Only 50% of sentences are marked with having entities in it, which makes the whole paragraph broken before calculating any of the metric. (But some sentences like "Uh-huh." should be igonred since it really doesn't contain entities.)
2. How to adjust to Conversations (Multiple speakers)? e.g. Document2

### Centering-based metrics of coherence

In [303]:
from collections import Counter 
def statistics(centering_document):
    cnt = Counter()
    cnt["num_all_u"] = len(centering_document)
    cnt["num_valid_u"], cnt["num_nocb"], cnt["num_cheapness"], cnt["num_coherence"], cnt["num_salience"], cnt[
        "num_continue"], cnt["num_retain"], cnt["num_s_shift"], cnt["num_r_shift"] = 0,0,0,0,0,0,0,0,0
    for centeringUtterance in centering_document:
        if len(centeringUtterance.candidate_entities) == 0:
            continue
        cnt["num_valid_u"] += 1
        if centeringUtterance.CB==None:
            cnt["num_nocb"] += 1
        if centeringUtterance.cheapness:
            cnt["num_cheapness"] += 1
        if centeringUtterance.coherence:
            cnt["num_coherence"] += 1
        if centeringUtterance.salience:
            cnt["num_salience"] += 1
        if centeringUtterance.transition == Transition.CONTIUNE:
            cnt["num_continue"] += 1
        elif centeringUtterance.transition == Transition.RETAIN:
            cnt["num_retain"] += 1
        elif centeringUtterance.transition == Transition.S_SHIFT:
            cnt["num_s_shift"] += 1
        elif centeringUtterance.transition == Transition.R_SHIFT:
            cnt["num_r_shift"] += 1
    df = pd.DataFrame([cnt.values()], columns=cnt.keys())
    return df

def more_stastics(df):
    '''
    for all the scores, higher == more coherence
    '''
    df['%valid_u'] = df['num_valid_u']/df['num_all_u']*100
    df['%not_nocb'] = (1-df['num_nocb']/df['num_valid_u'])*100
    df['%cheapness'] = df['num_cheapness']/df['num_valid_u']*100
    df['%coherence'] = df['num_coherence']/df['num_valid_u']*100
    df['%salience'] = df['num_salience']/df['num_valid_u']*100
    df['%continue'] = df['num_continue']/(df['num_valid_u']-1)*100
    df['%retain'] = df['num_retain']/(df['num_valid_u']-1)*100
    df['%s_shift'] = df['num_s_shift']/(df['num_valid_u']-1)*100
    df['%r_shift'] = df['num_r_shift']/(df['num_valid_u']-1)*100
    df['%kp'] = (df['%not_nocb'] + df['%cheapness']+ df['%coherence'] + df['%salience']) /4


In [341]:
pd.options.display.float_format = '{:.2f}'.format
result = pd.concat([statistics(centering_document) for centering_document in centering_documents], ignore_index=True, sort=False)
more_stastics(result)
result.describe()


,num_all_u,num_valid_u,num_nocb,num_cheapness,num_coherence,num_salience,num_continue,num_retain,num_s_shift,num_r_shift,%valid_u,%not_nocb,%cheapness,%coherence,%salience,%continue,%retain,%s_shift,%r_shift,%kp
count,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00
mean,23.00,13.50,5.00,5.50,7.00,6.00,5.00,2.00,1.00,4.50,50.33,50.00,22.92,43.75,39.58,44.57,8.70,4.35,42.39,39.06
std,21.21,14.85,4.24,7.78,8.49,7.07,5.66,2.83,1.41,4.95,18.14,23.57,32.41,14.73,8.84,7.69,12.30,6.15,10.76,19.89
min,8.00,3.00,2.00,0.00,1.00,1.00,1.00,0.00,0.00,1.00,37.50,33.33,0.00,33.33,33.33,39.13,0.00,0.00,34.78,25.00
25%,15.50,8.25,3.50,2.75,4.00,3.50,3.00,1.00,0.50,2.75,43.91,41.67,11.46,38.54,36.46,41.85,4.35,2.17,38.59,32.03
50%,23.00,13.50,5.00,5.50,7.00,6.00,5.00,2.00,1.00,4.50,50.33,50.00,22.92,43.75,39.58,44.57,8.70,4.35,42.39,39.06
75%,30.50,18.75,6.50,8.25,10.00,8.50,7.00,3.00,1.50,6.25,56.74,58.33,34.38,48.96,42.71,47.28,13.04,6.52,46.20,46.09
max,38.00,24.00,8.00,11.00,13.00,11.00,9.00,4.00,2.00,8.00,63.16,66.67,45.83,54.17,45.83,50.00,17.39,8.70,50.00,53.12


### Exploring the space of possible orderings

- Basis for Comparison(BfC)
- Step 1: to search through the space of possible orderings defined by the **permutations** of the CF lists that B consists of (using Random sampling).
- Step 2: to divide the explored search space into sets of orderings that score better, equal, or worse than B according to M.
- Step 3: $v(M,S)= Worse(M) + Equal(M)/2$.

In [342]:
import random


def Doc2CenterDoc(document):
    centeringUtterance = CenteringUtterance(document[0], 0, gram_role=gram_roles[i][0])
    centering_document = [centeringUtterance]
    for j in range(1, len(document)):
        prviousUtterance = centeringUtterance
        centeringUtterance = CenteringUtterance(document[j], j, gram_role=gram_roles[i][j])
        centeringUtterance.set_CB_and_cheapness_coherence(centering_document)
        centeringUtterance.set_salience()
        centeringUtterance.set_transition()
        centering_document.append(centeringUtterance)
    return centering_document


def shuffle_list(some_lists):
    randomized_list = some_lists[-1][:]
    flag = False
    while True:
        random.shuffle(randomized_list)
        for some_list in some_lists:
            if randomized_list == some_list:
                break
        else:
            return randomized_list


def get_centering_search_spaces(documents, search_space_size):
    centering_search_spaces = []
    for i, document in enumerate(documents):
        centering_search_space = [
            Doc2CenterDoc(document)]  # the centered original_document as centering_search_space[0]
        doc_search_spaces = [document]
        for k in range(1, search_space_size, 1):
            random_document = shuffle_list(doc_search_spaces)
            doc_search_spaces.append(random_document)
            centering_search_space.append(Doc2CenterDoc(random_document))
        centering_search_spaces.append(centering_search_space)
    return centering_search_spaces


def compute_classification_rate(centering_search_space, search_space_size, M: str):
    '''
    :param centering_search_space:  a list of random sampled centering documents from one original document
    :param search_space_size: len(centering_search_space)
    :param M: string, the metric, '%not_nocb' or '%cheapness' or 'transition'
                                  'kp'(sum up '%not_nocb'&'%cheapness'&'%coherence'&'%salience')
    :return: float [0,1], the classification rate, the higher, the better
    '''
    original_doc_score = centering_search_space[0]
    df = pd.concat([statistics(centering_document) for centering_document in centering_search_space], ignore_index=True,
                   sort=False)
    more_stastics(df)
    if M == 'transition':
        return compute_classification_rate_by_transition(df,search_space_size)
    worse = df[df[M] < df[M].get(0)].count()[0]
    equal = df[df[M] == df[M].get(0)].count()[0]
    classification_rate = (worse + equal / 2) / search_space_size
    return classification_rate


def compute_classification_rate_by_transition(df,search_space_size):
    worse = df[(df['%continue'] < df['%continue'].get(0))
               | ((df['%continue'] == df['%continue'].get(0)) & (df['%retain'] < df['%retain'].get(0))) |
               ((df['%continue'] == df['%continue'].get(0)) & (df['%retain'] == df['%retain'].get(0)) & (
                           df['%s_shift'] < df['%s_shift'].get(0)))
               ].count()[0]
    equal = df[((df['%continue'] == df['%continue'].get(0)) & (df['%retain'] == df['%retain'].get(0)) & (
                df['%s_shift'] == df['%s_shift'].get(0)))
    ].count()[0]
    classification_rate = (worse + equal / 2) / search_space_size
    return classification_rate

In [337]:
search_space_size = 100
centering_search_spaces = get_centering_search_spaces(documents, search_space_size)
metrics = ['%not_nocb', '%cheapness', '%coherence', '%salience', '%kp', 'transition']
aver_classif_rates=dict()
for M in metrics: 
    classification_rates = [compute_classification_rate(centering_search_space, search_space_size, M) for centering_search_space in centering_search_spaces]
    aver_classif_rate = sum(classification_rates)/len(classification_rates) * 100
    aver_classif_rates[M] = aver_classif_rate

df_aver_classif_rates = pd.DataFrame.from_dict(aver_classif_rates, orient='index').reset_index()
df_aver_classif_rates


{'%not_nocb': 50.24999999999999, '%cheapness': 51.75000000000001, '%coherence': 50.24999999999999, '%salience': 41.5, '%kp': 49.5, 'transition': 41.5}


,index,0
0,%not_nocb,50.25
1,%cheapness,51.75
2,%coherence,50.25
3,%salience,41.50
4,%kp,49.50
5,transition,41.50


## Run on Test Dataset

In [344]:
!python /cluster/project/infk/cotterell/ct/code/centering_exp/corpusAnalysis.py

       num_all_u  num_valid_u  num_nocb  num_cheapness  num_coherence  \
count     348.00       348.00    348.00         348.00         348.00   
mean       27.24        21.47      7.63           9.84          10.42   
std        23.47        18.22      6.40          10.04          10.05   
min         2.00         0.00      0.00           0.00           0.00   
25%        10.00         7.00      2.00           2.00           3.00   
50%        20.00        15.00      6.00           6.00           7.00   
75%        38.00        31.25     11.00          14.00          15.00   
max       140.00        92.00     31.00          58.00          56.00   

       num_salience  num_continue  num_retain  num_s_shift  num_r_shift  \
count        348.00        348.00      348.00       348.00       348.00   
mean           9.60          7.34        3.08         2.26         7.80   
std            9.52          7.43        3.38         2.75         7.14   
min            0.00          0.00        0

# Analysis the Prediction of c2f-coref Model

In [ ]:
import os 
os.chdir('/cluster/project/infk/cotterell/ct/code/centering_exp')
from readOntoNotes import *
from centeringUtterance import *


## Get predicted_cluster_ids (by Run Kenton Lee’s model w/ Spanbert embedding)

**Get all indexes of entities (we need to add new entities from predicted clusters into the dictionary)**

- from: 
    - 'clusters': [[[0, 1], [25, 25], [37, 37], [48, 48], [55, 55]], [[11, 14], [34, 34]], [[39, 53], [57, 57]], [[55, 57], [63, 63], [72, 72]]]}
    - 'top_spans': [[0, 1], [0, 14], [3, 3], [5, 14], [8, 8], [11, 13], [11, 14], [13, 13], [16, 18], [16, 22], [20, 22], [24, 24], [25, 25], [27, 35], [27, 53], [34, 34], [37, 37], [39, 53], [48, 48], [49, 49], [50, 53], [55, 55], [55, 57], [57, 57], [58, 58], [59, 61], [59, 69], [63, 63], [63, 69], [71, 71], [72, 72], [72, 74], [76, 81]],  and
    - 'predicted_antecedents': [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 0, -1, -1, 6, 12, -1, 12, -1, -1, 0, -1, 17, -1, -1, -1, 22, -1, -1, 22, -1, -1]
    - doc_id, list[list[int]]   [[0,1,2,3,...],[14,15,16,...],[], ...]
- to:
    - candidate_entities per sentence: [(42, (5, 9)), (32, (11, 13)), (42, (2, 2))]

In [406]:
all_entities = set()
for i, document in enumerate(documents):
    for sentence in document:
        entity_ids = [coref_span[0] for coref_span in sentence.coref_spans]
        all_entities.update(entity_ids)
max_entity_id = max(all_entities)

In [416]:
predicted_candidate_entities = []
for i, centering_document in enumerate(centering_documents):
    doc_id_list, predicted_dict = doc_ids[i], predicted_dicts[i]
    predicted_candidate_entities_doc = [[] for i in range(len(doc_id_list))]
    current_u_id, current_total_id, entity_id = 0, 0, max_entity_id+1
    for j, top_span in enumerate(predicted_dict['top_spans']):
        if current_u_id == len(doc_id_list):
            break
        while not current_u_id == len(doc_id_list) and not (current_total_id <= top_span[0] and top_span[1] < current_total_id + len(doc_id_list[current_u_id])):
            current_total_id += len(doc_id_list[current_u_id])
            current_u_id += 1
        if current_total_id <= top_span[0] and top_span[1] < current_total_id + len(doc_id_list[current_u_id]):
            # this top_span is in the current_u_id
            local_span_id_pair = (entity_id, (top_span[0]-current_total_id, top_span[1]-current_total_id))
            entity_id +=1
            predicted_candidate_entities_doc[current_u_id].append(local_span_id_pair)
    predicted_candidate_entities.append(predicted_candidate_entities_doc)
    
predicted_candidate_entities[1]



[[(121, (0, 0)), (122, (3, 4))],
 [(123, (3, 4))],
 [(124, (1, 1)),
  (125, (4, 4)),
  (126, (5, 5)),
  (127, (6, 6)),
  (128, (6, 7)),
  (129, (9, 21)),
  (130, (16, 16)),
  (131, (18, 18)),
  (132, (20, 21))],
 [(133, (4, 5)),
  (134, (4, 6)),
  (135, (7, 11)),
  (136, (12, 12)),
  (137, (12, 35)),
  (138, (14, 29)),
  (139, (14, 35)),
  (140, (20, 21)),
  (141, (20, 29)),
  (142, (23, 26)),
  (143, (28, 29)),
  (144, (31, 31)),
  (145, (33, 34)),
  (146, (33, 35)),
  (147, (35, 35))],
 [(148, (1, 5)), (149, (1, 6)), (150, (4, 5)), (151, (7, 7)), (152, (8, 10))],
 [(153, (1, 4)), (154, (5, 5)), (155, (10, 12))],
 [(156, (1, 1)),
  (157, (5, 12)),
  (158, (11, 12)),
  (159, (13, 13)),
  (160, (16, 17))],
 [(161, (2, 3)), (162, (10, 10)), (163, (11, 12)), (164, (11, 14))],
 [(165, (2, 2)),
  (166, (7, 8)),
  (167, (9, 9)),
  (168, (11, 11)),
  (169, (12, 21)),
  (170, (12, 31)),
  (171, (15, 16)),
  (172, (18, 18)),
  (173, (21, 21)),
  (174, (26, 26)),
  (175, (28, 31)),
  (176, (31, 

In [418]:
len(predicted_dicts[1]['document'])

564

In [415]:
doc_ids[1]

[[0, 1, 2, 3, 4],
 [5, 6, 7, 8, 9],
 [10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31],
 [32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68],
 [69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79],
 [80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92],
 [93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110],
 [111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125],
 [126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  156,
  157,
  158,
  159,
  160],
 [161,
  162,
  163,
  164,
  165,
  166,
  167,
  168,
  169,

In [1]:
# predicted_dicts的tokens数量与ontonote document里的不一样！
# 可以输出predicted_dicts[1]['document']和document里所有sentence的tokens，看看区别